In [0]:
%sql

CREATE OR REPLACE TABLE `drive-to-survive`.default.scored_aggregate AS
SELECT 
  Name, 
  Round,
  Race,
  Driver,
  constructor_name AS Constructor,
  SUM(gp) AS total_gp,
  SUM(gp_expected) AS total_gp_expected,
  SUM(gp_var) AS total_gp_var,
  SUM(fastest_lap) AS total_fastest_lap,
  SUM(COALESCE(sprint, 0)) AS total_sprint,
  SUM(COALESCE(sprint_expected, 0)) AS total_sprint_expected,
  SUM(COALESCE(sprint_var, 0)) AS total_sprint_var,
  SUM(gp) + SUM(fastest_lap) + SUM(COALESCE(sprint, 0)) AS total,
  SUM(gp_expected) + SUM(COALESCE(sprint_expected, 0)) AS total_expected,
  SUM(gp_var) + SUM(COALESCE(sprint_var, 0)) AS total_var,
  SUM(SUM(gp) + SUM(fastest_lap) + SUM(COALESCE(sprint, 0))) OVER (
    PARTITION BY Name 
    ORDER BY Round 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_total,
  SUM(SUM(gp_expected) + SUM(COALESCE(sprint_expected, 0))) OVER (
    PARTITION BY Name 
    ORDER BY Round 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_expected,
  SUM(SUM(gp_var) + SUM(COALESCE(sprint_var, 0))) OVER (
    PARTITION BY Name 
    ORDER BY Round 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_var
FROM `drive-to-survive`.default.picks_scored
GROUP BY Name, Round, Race, Driver, constructor_name
ORDER BY Round, cumulative_total, Name